In [59]:
import os
import sys
import time
import yaml
import pandas as pd
import numpy as np
import re
import subprocess

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
R_PATH = local_config['R_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import writing_tools as wt
from utils import parse_casenum
import utils

from matplotlib import pyplot as plt
from scipy.spatial.distance import mahalanobis


plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)
with open('../../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']
EMBEDDING_DIMENSION = config['EMBEDDING_DIMENSION']

rng = np.random.default_rng(12898)

N_CLUSTERS = 3
N_COMPONENTS = 10


In [60]:
main_df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "ologit_regression_data.parquet"))
main_df['canonical_casenum'] = main_df['title'].apply(lambda x: parse_casenum(x)['canonical_casenum'])
main_df = main_df.sort_values(by=['canonical_casenum', 'date'], ascending=True).reset_index(drop=True)
main_df['times_appeared'] = main_df.groupby('canonical_casenum').cumcount() + 1

In [61]:
df = []
for idx, row in main_df.iterrows():
    if idx==0:
        continue
    prev_row = main_df.iloc[idx-1]
    if prev_row['canonical_casenum'] != row['canonical_casenum']:
        continue
    if prev_row['project_result'] != "DELAYED":
        continue
    if row['project_result'] == "APPLICATION WITHDRAWN":
        continue
    days_between = (pd.to_datetime(row['date']) - pd.to_datetime(prev_row['date'])).days
    prev_members_present = set(list(prev_row['ayes']) + list(prev_row['noes']) + list(prev_row['abstentions']) + list(prev_row['recusals']))
    members_present = set(list(row['ayes']) + list(row['noes']) + list(row['abstentions']) + list(row['recusals']))
    if len(prev_members_present) == 0:
        continue
    frac_new = len(prev_members_present - members_present) / len(prev_members_present)
    new_row = row.copy()
    new_row['days_between'] = days_between
    new_row['frac_new'] = frac_new
    df.append(new_row)

df = pd.DataFrame(df).reset_index(drop=True)
df['outcome'] = 0
df.loc[df['project_result'] == "DELAYED", 'outcome'] = 1

df.to_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "rehearing_delays.parquet"))

In [63]:
df

,date,year,item_no,sub_item_no,item_order_changed,taken_off_consent_calendar,multiple_votes,mover,seconder,ayes,...,is_nonresidential,log2_support,log2_oppose,log_square_footage,height_missing,log_square_footage_missing,canonical_casenum,times_appeared,days_between,frac_new
0,2018-10-25,2018,10,,False,False,False,Ambroz,Khorsand,"[Ambroz, Khorsand, Choe, Mack, Millman, Dake W...",...,True,0.0,1.584963,12.203575,False,False,AA-2017-397,2,28,0.125000
1,2024-10-24,2024,7,,False,False,False,Newhouse,Diaz,"[Newhouse, Diaz, Klein, Lawshe, Mack, Saitman]",...,False,0.0,0.000000,0.000000,True,True,AA-2022-8610,2,42,0.400000
2,2019-02-28,2019,6,,False,False,False,Khorsand,Ambroz,"[Khorsand, Ambroz, Mack, Millman, Mitchell]",...,False,0.0,0.000000,0.000000,True,True,CPC-2008-3470,2,217,0.285714
3,2022-02-10,2022,7,,True,False,False,Millman,Hornstock,"[Millman, Hornstock, Leung, López-Ledesma, Mac...",...,False,0.0,0.000000,0.000000,True,True,CPC-2013-1495,2,154,0.142857
4,2022-03-10,2022,8,,False,False,False,López-Ledesma,Campbell,"[López-Ledesma, Campbell, Choe, Leung, Mack, P...",...,False,0.0,0.000000,0.000000,True,True,CPC-2013-1495,3,28,0.285714
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,2022-04-28,2022,9,,False,False,False,Perlman,Millman,"[Perlman, Millman, Campbell, Hornstock, Leung,...",...,True,1.0,4.087463,7.824446,False,False,ZA-2017-3950,2,49,0.285714
106,2022-08-11,2022,9,,False,False,False,Millman,Hornstock,"[Millman, Hornstock, Campbell, López-Ledesma, ...",...,True,1.0,2.000000,7.824446,True,False,ZA-2017-3950,3,105,0.142857
107,2022-08-11,2022,11,,False,False,False,Millman,Hornstock,"[Campbell, López-Ledesma, Mack, Perlman, Dake ...",...,True,1.0,2.000000,0.000000,True,True,ZA-2017-3953,2,105,0.428571
108,2022-12-15,2022,8,,True,False,False,Perlman,López-Ledesma,"[Perlman, López-Ledesma, Campbell, Hornstock, ...",...,False,0.0,2.000000,9.952135,False,False,ZA-2021-6672,2,7,0.200000
